In [2]:
from pathlib import Path
import pandas as pd
import random
import shutil
import yaml
from ultralytics import YOLO
import cv2
import os
import torch
import torch.utils

In [3]:
# grab data

data_path = Path('C:/Users/sampl/Documents/code/school/aiep/aiep_project/data')

image_path = data_path / 'images'
label_path = data_path / 'labels'

processed_data_path = Path('C:/Users/sampl/Documents/code/school/aiep/aiep_project/processed_data')
train_path = processed_data_path / 'train'
train_images_path = train_path / 'images'
train_labels_path = train_path / 'labels'

val_path = processed_data_path / 'val'
val_images_path = val_path / 'images'
val_labels_path = val_path / 'labels'

output_images_path = processed_data_path / 'output'

# delete files
if Path.exists(processed_data_path):
    shutil.rmtree(processed_data_path)


# make files
os.mkdir(processed_data_path)
os.mkdir(train_path)
os.mkdir(train_images_path)
os.mkdir(train_labels_path)
os.mkdir(val_path)
os.mkdir(val_images_path)
os.mkdir(val_labels_path)

In [4]:
image_file_list = [path for path in Path(image_path).rglob('*')]
label_file_list = [path for path in Path(label_path).rglob('*')]

print(len(image_file_list))

95


In [5]:
train_percent = 0.8

file_num = len(image_file_list)

train_num = int(train_percent * file_num)
val_num = file_num - train_num

print(train_num)
print(val_num)

76
19


In [6]:
# distribute data

for i, num in enumerate([train_num, val_num]):

    for index in range(num):

        img_path = random.choice(image_file_list)
        image_name = img_path.name
        base_name = img_path.stem
        text_name = base_name + '.txt'
        text_path = Path(label_path / text_name)

        # train
        if i == 0:
            shutil.copy(img_path, train_images_path)
            shutil.copy(text_path, train_labels_path)

        # val
        else:
            shutil.copy(img_path, val_images_path)
            shutil.copy(text_path, val_labels_path)

In [7]:
# create yaml files

class_path = data_path / 'classes.txt'
data_yaml_path = processed_data_path / 'data.yaml'

data = {
    'path' : str(processed_data_path),
    'train' : 'train/images',
    'val' : 'val/images',
    'nc' : 1,
    'names' : ['Flosspick']
}

# write to yaml
with open(data_yaml_path, 'w') as file:
    yaml.dump(data, file, sort_keys = False)

In [8]:
# load pretrained model

pretrained_path = Path(r'C:\Users\sampl\Documents\code\school\aiep\aiep_project\src\aiep_project\yolo26s.pt')

model = YOLO(pretrained_path)

# train
results = model.train(
    data = data_yaml_path,
    epochs = 60,
    imgsz = 640,
    device = 0
)

New https://pypi.org/project/ultralytics/8.4.215 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.210  Python-3.14.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\sampl\Documents\code\school\aiep\aiep_project\processed_data\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, ma

In [9]:
# test model

model = YOLO(r'C:\Users\sampl\Documents\code\school\aiep\aiep_project\runs\detect\train\weights\best.pt')

train_results = model.predict(
    source = r'C:/Users/sampl/Documents/code/school/aiep/aiep_project/processed_data/train/images',
    conf = 0.5,
    save = True,
    save_crop = True
)





image 1/55 C:\Users\sampl\Documents\code\school\aiep\aiep_project\processed_data\train\images\05f67048-1794459999252.jpg: 640x480 1 Flosspick, 15.4ms
image 2/55 C:\Users\sampl\Documents\code\school\aiep\aiep_project\processed_data\train\images\0701982d-1794459997730.jpg: 640x480 1 Flosspick, 7.3ms
image 3/55 C:\Users\sampl\Documents\code\school\aiep\aiep_project\processed_data\train\images\074e01ca-1794459997538.jpg: 640x480 1 Flosspick, 6.7ms
image 4/55 C:\Users\sampl\Documents\code\school\aiep\aiep_project\processed_data\train\images\090374b9-1794459999083.jpg: 640x480 1 Flosspick, 9.3ms
image 5/55 C:\Users\sampl\Documents\code\school\aiep\aiep_project\processed_data\train\images\0b261c90-1794459998119.jpg: 640x480 1 Flosspick, 9.2ms
image 6/55 C:\Users\sampl\Documents\code\school\aiep\aiep_project\processed_data\train\images\0dc3ebd3-1794459998195.jpg: 640x480 1 Flosspick, 7.2ms
image 7/55 C:\Users\sampl\Documents\code\school\aiep\aiep_project\processed_data\train\images\12f3c7ff-1

In [10]:
# grab data

data_path = Path('C:/Users/sampl/Documents/code/school/aiep/aiep_project/cropped_data')

image_path = data_path / 'images'
label_path = data_path / 'labels'

processed_data_path = Path('C:/Users/sampl/Documents/code/school/aiep/aiep_project/cropped_processed_data')
train_images_path = processed_data_path / 'train/images'
train_labels_path = processed_data_path / 'train/labels'

val_images_path = processed_data_path / 'val/images'
val_labels_path = processed_data_path / 'val/labels'

output_images_path = processed_data_path / 'output'

In [11]:
image_file_list = [path for path in Path(image_path).rglob('*')]
label_file_list = [path for path in Path(label_path).rglob('*')]

print(len(image_file_list))

0


In [12]:
train_percent = 0.8

file_num = len(image_file_list)

train_num = int(train_percent * file_num)
val_num = file_num - train_num

print(train_num)
print(val_num)

0
0


In [13]:
# distribute data

for i, num in enumerate([train_num, val_num]):

    for index in range(num):

        img_path = random.choice(image_file_list)
        image_name = img_path.name
        base_name = img_path.stem
        text_name = base_name + '.txt'
        text_path = Path(label_path / text_name)

        # train
        if i == 0:
            shutil.copy(img_path, train_images_path)
            shutil.copy(text_path, train_labels_path)

        # val
        else:
            shutil.copy(img_path, val_images_path)
            shutil.copy(text_path, val_labels_path)

In [14]:
# create yaml files

class_path = data_path / 'classes.txt'
data_yaml_path = processed_data_path / 'data.yaml'

data = {
    'path' : str(processed_data_path),
    'train' : 'train/images',
    'val' : 'val/images',
    'nc' : 2,
    'names' : ['Flosspick', 'String']
}

# write to yaml
with open(data_yaml_path, 'w') as file:
    yaml.dump(data, file, sort_keys = False)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\sampl\\Documents\\code\\school\\aiep\\aiep_project\\cropped_processed_data\\data.yaml'

In [ ]:
# load pretrained model

pretrained_crop_path = Path(r'C:\Users\sampl\Documents\code\school\aiep\aiep_project\src\aiep_project\yolo26s-seg.pt')

model_crop = YOLO(pretrained_crop_path)

# train
results = model_crop.train(
    data = data_yaml_path,
    epochs = 60,
    imgsz = 640,
    device = 0
)

New https://pypi.org/project/ultralytics/8.4.215 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.210  Python-3.14.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\sampl\Documents\code\school\aiep\aiep_project\cropped_processed_data\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=